In [1]:
!pip install qutip

In [2]:
import numpy as np
import pandas as pd
import qutip as qt

print(qt.__version__)
print(pd.__version__)

5.2.0
2.2.2


In [2]:
"""
QuTiP notebook-friendly ED time evolution for the Hamiltonian

    H = sum_{j=0}^{N-2} Jz * σ^z_j σ^z_{j+1}
      + sum_{i<j} (Jx / |j-i|^alpha) * σ^x_i σ^x_j

Utilities compute observables and save them to CSV.
"""

from __future__ import annotations
from typing import List, Optional, Tuple, Dict
import os

import numpy as np
import pandas as pd
import qutip as qt


# -----------------------------
# Operator constructors
# -----------------------------

def op_on_site(op: qt.Qobj, N: int, site: int) -> qt.Qobj:
    if N <= 0:
        raise ValueError("N must be positive.")
    if not (0 <= site < N):
        raise ValueError(f"site={site} out of range for N={N}.")
    ops = [qt.qeye(2)] * N
    ops[site] = op
    return qt.tensor(ops)


def two_site_op(opA: qt.Qobj, opB: qt.Qobj, N: int, i: int, j: int) -> qt.Qobj:
    if N <= 0:
        raise ValueError("N must be positive.")
    if i == j:
        raise ValueError("i and j must be different sites.")
    if not (0 <= i < N and 0 <= j < N):
        raise ValueError(f"Sites (i={i}, j={j}) out of range for N={N}.")
    ops = [qt.qeye(2)] * N
    ops[i] = opA
    ops[j] = opB
    return qt.tensor(ops)


# -----------------------------
# Hamiltonian
# -----------------------------

def kac_factor(N: int, alpha: float) -> float:
    """
    Returns Kac factor K such that Jx_eff = Jx / K.
    Matches: K = sum_{i<j} 1/r^alpha divided by (N-1).
    """
    r = np.arange(1, N, dtype=float)
    # sum_{i<j} 1/r^alpha = sum_{r=1}^{N-1} (N-r)/r^alpha
    sumV = np.sum((N - r) / (r ** alpha))
    return sumV / (N - 1)


def build_H(N: int, Jz: float, Jx: float, alpha: float, kac: bool = True) -> qt.Qobj:
    if N < 2:
        raise ValueError("N must be >= 2.")
    if alpha < 0:
        raise ValueError("alpha must be >= 0.")

    sx, sz = qt.sigmax(), qt.sigmaz()
    H = 0

    if kac:
        K = kac_factor(N, alpha)
        Jx = Jx / K  # rescaled long-range prefactor

    # Nearest-neighbor zz
    for j in range(N - 1):
        H += Jz * two_site_op(sz, sz, N, j, j + 1)

    # xx term
    if alpha >= 100:
        for j in range(N - 1):
            H += Jx * two_site_op(sx, sx, N, j, j + 1)
    else:
        for i in range(N):
            for j in range(i + 1, N):
                r = j - i
                H += (Jx / (r ** alpha)) * two_site_op(sx, sx, N, i, j)

    return H

import numpy as np
import qutip as qt

def build_Heff(
    N: int,
    Jz: float,
    Jx: float,
    alpha: float,
    kac: bool = True,
    periodic_nn: bool = False,
) -> qt.Qobj:
    """
    Implements Eq. (0thOrder):

    H0 =
      Jz * sum_j ( ad_j a_{j+1} + ad_{j+1} a_j + ad_j ad_{j+1} + a_j a_{j+1} )
    - 2 * sum_{i<j} (Jx/|i-j|^alpha) * (n_i + n_j)
    + 4 * sum_{i<j} (Jx/|i-j|^alpha) * n_i n_j
    +     sum_{i<j} (Jx/|i-j|^alpha)   [constant shift]

    Hard-core via local dim=2: a=sigmam(), ad=sigmap(), n=num(2).
    """

    if N < 2:
        raise ValueError("N must be >= 2.")
    if alpha < 0:
        raise ValueError("alpha must be >= 0.")

    a  = qt.sigmam()
    ad = qt.sigmap()
    n  = qt.num(2)  # |1><1| in hard-core basis

    # Kac scaling only for LR part (Jx terms), same convention as your earlier code
    Jx_eff = Jx
    if kac:
        K = kac_factor(N, alpha)
        Jx_eff = Jx / K

    H = 0 * op_on_site(qt.qeye(2), N, 0)
    I_full = qt.tensor([qt.qeye(2) for _ in range(N)])

    # -----------------------------
    # NN Jz term (open bonds)
    # -----------------------------
    for j in range(N - 1):
        #hopping
        H += Jz * (two_site_op(ad, a,  N, j, j + 1) + two_site_op(ad, a, N, j + 1, j))
        #pairing
        # H += Jz * (two_site_op(ad, ad, N, j, j + 1) + two_site_op(a,  a,  N, j, j + 1))

    if periodic_nn:
        j, jp = N - 1, 0
        H += Jz * (two_site_op(ad, a,  N, j, jp) + two_site_op(ad, a, N, jp, j))
        H += Jz * (two_site_op(ad, ad, N, j, jp) + two_site_op(a,  a,  N, j, jp))

    # -----------------------------
    # LR terms exactly as written
    # -----------------------------
    def g(r: int) -> float:
        return 0.0 if r == 0 else 1.0 / (r ** alpha)

    for i in range(N):
        for j in range(i + 1, N):
            r = j - i
            gr = g(r)
            if gr == 0.0:
                continue

            Jij = Jx_eff * gr

            # -2 * Jij * (n_i + n_j)
            H += (-2.0 * Jij) * op_on_site(n, N, i)
            H += (-2.0 * Jij) * op_on_site(n, N, j)

            # +4 * Jij * n_i n_j
            H += (4.0 * Jij) * two_site_op(n, n, N, i, j)

            # + Jij * I
            H += Jij * I_full

    return H


import numpy as np
import qutip as qt

def build_Heff2(
    N: int,
    Jz: float,
    Jlong: float,
    alpha: float,
    kac: bool = True,
    periodic_nn: bool = False,
) -> qt.Qobj:
    """
    Implements Eq. (effham2):

    H =
      Jz * sum_{j=0}^{N-2} [ 1 - 2(n_j + n_{j+1}) + 4 n_j n_{j+1} ]
    + sum_{i<j} (Jlong / |i-j|^alpha) * (
          b_j^† b_i + b_i^† b_j
        + b_j^† b_i^† + b_j b_i
      )

    Hard-core bosons via local dim=2:
      b  = sigmam(),  b^† = sigmap(),  n = num(2).
    """

    if N < 2:
        raise ValueError("N must be >= 2.")
    if alpha < 0:
        raise ValueError("alpha must be >= 0.")

    # hard-core boson operators (spin-1/2 mapping)
    b  = qt.sigmam()
    bd = qt.sigmap()
    n  = qt.num(2)

    # optional Kac rescale for LR term
    Jlong_eff = Jlong
    if kac:
        K = kac_factor(N, alpha)   # assumes you already have this
        Jlong_eff = Jlong / K

    # start with zero operator with correct dims
    H = 0 * op_on_site(qt.qeye(2), N, 0)  # assumes you already have op_on_site

    I_full = qt.tensor([qt.qeye(2) for _ in range(N)])

    # --------------------------------------------------
    # 1) NN Jz term: sum_j [ 1 - 2(nj+nj+1) + 4 nj nj+1 ]
    # --------------------------------------------------
    # open boundary bonds: (j, j+1) for j=0..N-2
    for j in range(N - 1):
        # +1 term
        H += Jz * I_full

        # -2(n_j + n_{j+1})
        H += (-2.0 * Jz) * op_on_site(n, N, j)
        H += (-2.0 * Jz) * op_on_site(n, N, j + 1)

        # +4 n_j n_{j+1}
        H += (4.0 * Jz) * two_site_op(n, n, N, j, j + 1)  # assumes you have two_site_op

    if periodic_nn:
        j, jp = N - 1, 0

        H += Jz * I_full
        H += (-2.0 * Jz) * op_on_site(n, N, j)
        H += (-2.0 * Jz) * op_on_site(n, N, jp)
        H += (4.0 * Jz) * two_site_op(n, n, N, j, jp)

    # --------------------------------------------------
    # 2) LR term: sum_{i<j} Jlong/r^alpha * (hop + pairing)
    # --------------------------------------------------
    def g(r: int) -> float:
        return 0.0 if r == 0 else 1.0 / (r ** alpha)

    for i in range(N):
        for j in range(i + 1, N):
            r = j - i
            gr = g(r)
            if gr == 0.0:
                continue

            Jij = Jlong_eff * gr

            # hopping: bd_j b_i + bd_i b_j
            H += Jij * (two_site_op(bd, b, N, j, i) + two_site_op(bd, b, N, i, j))

            # pairing: bd_j bd_i + b_j b_i
            H += Jij * (two_site_op(bd, bd, N, j, i) + two_site_op(b, b, N, j, i))

    return H

# -----------------------------
# Initial states
# -----------------------------

def superpose(psi1: qt.Qobj, psi2: qt.Qobj, normalize: bool = True) -> qt.Qobj:
    if not (psi1.isket and psi2.isket):
        raise ValueError("Both inputs must be kets.")
    if psi1.shape != psi2.shape:
        raise ValueError("States have different Hilbert-space dimensions.")
    if psi1.dims != psi2.dims:
        raise ValueError("States have incompatible dims; fix before superposing.")
    psi = psi1 + psi2
    return psi.unit() if normalize else psi / np.sqrt(2)


def product_state_from_bitstring(bitstring: str) -> qt.Qobj:
    s = bitstring.strip().replace(" ", "")
    if not s or any(c not in "01" for c in s):
        raise ValueError("bitstring must be a non-empty string of 0/1 characters.")
    kets = [qt.basis(2, int(c)) for c in s]
    return qt.tensor(kets).unit()


def product_state_angles(thetas: np.ndarray, phis: np.ndarray) -> qt.Qobj:
    thetas = np.asarray(thetas, dtype=float)
    phis = np.asarray(phis, dtype=float)
    if thetas.shape != phis.shape:
        raise ValueError("thetas and phis must have the same shape.")
    N = thetas.size
    if N == 0:
        raise ValueError("Need at least one site.")

    kets = []
    for th, ph in zip(thetas, phis):
        ket = np.cos(th / 2) * qt.basis(2, 0) + np.exp(1j * ph) * np.sin(th / 2) * qt.basis(2, 1)
        kets.append(ket)
    return qt.tensor(kets).unit()


def product_state_x_pm(pm: str) -> qt.Qobj:
    pm = pm.strip().replace(" ", "")
    if not pm or any(c not in "+-" for c in pm):
        raise ValueError("pm must be a non-empty string of '+'/'-'.")
    N = len(pm)
    thetas = np.full(N, np.pi / 2)  # equator
    phis = np.array([0.0 if c == "+" else np.pi for c in pm], dtype=float)
    return product_state_angles(thetas, phis)

def plus_minus_x() -> tuple[qt.Qobj, qt.Qobj]:
    """Return |+_x>, |-_x> expressed in the computational (z) basis."""
    up = qt.basis(2, 0)   # |0>_z
    dn = qt.basis(2, 1)   # |1>_z
    plus  = (up + dn).unit()
    minus = (up - dn).unit()
    return plus, minus

def two_minus_x_ket(N: int, i: int, j: int) -> qt.Qobj:
    """
    |i,j>_x := |-x> at sites i and j, and |+x> elsewhere (0-indexed).
    """
    if i == j:
        raise ValueError("i and j must be different.")
    if not (0 <= i < N and 0 <= j < N):
        raise ValueError(f"Sites (i={i}, j={j}) out of range for N={N}.")
    plus, minus = plus_minus_x()
    kets = [plus for _ in range(N)]
    kets[i] = minus
    kets[j] = minus
    return qt.tensor(kets).unit()

def one_excitation_ket(N: int, site: int) -> qt.Qobj:
    """|site> = |0...010...0> in computational (z) basis, 0-indexed."""
    kets = [qt.basis(2, 0) for _ in range(N)]
    kets[site] = qt.basis(2, 1)
    return qt.tensor(kets).unit()

def psi_23_superposition(N: int, theta: float, phi: float, site2: int = 1, site3: int = 2) -> qt.Qobj:
    """cos(theta)|2> + sin(theta) e^{i phi} |3> (sites are 0-indexed)."""
    ket2 = one_excitation_ket(N, site2)
    ket3 = one_excitation_ket(N, site3)
    psi = np.cos(theta) * ket2 + np.sin(theta) * np.exp(1j * phi) * ket3
    return psi.unit()

def two_excitation_superposition(
     N: int,
    site_fixed: int,
    site2: int,
    site3: int,
    theta: float,
    phi: float,
    pm: int = +1,
) -> qt.Qobj:
    """
    cos(theta)|site2, site_fixed>_z + pm * sin(theta) e^{i phi} |site3, site_fixed>_z

    where |a,b>_z means |1> on sites a,b and |0> elsewhere (computational basis).
    """
    # if site2 == site3:
    #     raise ValueError("site2 and site3 must be different.")
    # if len({site_fixed, site2, site3}) < 3:
    #     # site_fixed must be different from site2 and site3 as well
    #     raise ValueError("site_fixed must be different from site2 and site3.")
    # if not (0 <= site_fixed < N and 0 <= site2 < N and 0 <= site3 < N):
    #     raise ValueError("One or more site indices out of range.")

    # |site2, site_fixed> in z basis
    kets2 = [qt.basis(2, 0) for _ in range(N)]
    kets2[site_fixed] = qt.basis(2, 1)
    kets2[site2]      = qt.basis(2, 1)
    ket2 = qt.tensor(kets2).unit()

    # |site3, site_fixed> in z basis
    kets3 = [qt.basis(2, 0) for _ in range(N)]
    kets3[site_fixed] = qt.basis(2, 1)
    kets3[site3]      = qt.basis(2, 1)
    ket3 = qt.tensor(kets3).unit()

    psi = np.cos(theta) * ket2 + pm * np.sin(theta) * np.exp(1j * phi) * ket3
    return psi.unit()

def two_excitation_superposition_xpm(
    N: int,
    site_fixed: int,
    site2: int,
    site3: int,
    theta: float,
    phi: float,
    pm: int = +1,
) -> qt.Qobj:
    """
    cos(theta)|site2, site_fixed>_x + pm * sin(theta) e^{i phi} |site3, site_fixed>_x

    where |a,b>_x means |-x> on sites a,b and |+x> elsewhere.
    """
    ket2 = two_minus_x_ket(N, site2, site_fixed)
    ket3 = two_minus_x_ket(N, site3, site_fixed)
    psi = np.cos(theta) * ket2 + pm * np.sin(theta) * np.exp(1j * phi) * ket3
    return psi.unit()

# -----------------------------
# Dynamics
# -----------------------------

def evolve_state(H: qt.Qobj, psi0: qt.Qobj, times: np.ndarray) -> List[qt.Qobj]:
    times = np.asarray(times, dtype=float)
    res = qt.sesolve(H, psi0, times, e_ops=[])
    return res.states


# -----------------------------
# Observables
# -----------------------------

def magnetization_z(psi: qt.Qobj, N: int, sz_ops: Optional[List[qt.Qobj]] = None) -> np.ndarray:
    if sz_ops is None:
        sz = qt.sigmaz()
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]
    return np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)


def magnetization_x(psi: qt.Qobj, N: int, sx_ops: Optional[List[qt.Qobj]] = None) -> np.ndarray:
    if sx_ops is None:
        sx = qt.sigmax()
        sx_ops = [op_on_site(sx, N, i) for i in range(N)]
    return np.array([qt.expect(sx_ops[i], psi).real for i in range(N)], dtype=float)


def correlations_xx_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    xx_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sx_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    sx = qt.sigmax()
    if sx_ops is None:
        sx_ops = [op_on_site(sx, N, i) for i in range(N)]

    mx = np.array([qt.expect(sx_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = float(1.0 - mx[i] * mx[i])  # connected self correlator
            continue

        if xx_ops is None:
            op = two_site_op(sx, sx, N, i, j)
            xx = qt.expect(op, psi).real
        else:
            xx = qt.expect(xx_ops[(i, j)], psi).real

        out[(i, j)] = float(xx - mx[i] * mx[j])

    return out

def correlations_zz_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    zz_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sz_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    Connected zz correlator:
        Czz_ij = <sz_i sz_j> - <sz_i><sz_j>
    """
    sz = qt.sigmaz()
    if sz_ops is None:
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]

    mz = np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            # connected self-correlator = <sz^2> - <sz>^2 = 1 - mz^2  (since sz^2 = I)
            out[(i, j)] = float(1.0 - mz[i] * mz[i])
            continue

        if zz_ops is None:
            op = two_site_op(sz, sz, N, i, j)
            zz = qt.expect(op, psi).real
        else:
            zz = qt.expect(zz_ops[(i, j)], psi).real

        out[(i, j)] = float(zz - mz[i] * mz[j])

    return out


def concurrence_pair(psi: qt.Qobj, i: int, j: int) -> float:
    if i == j:
        return 0.0
    rho_ij = qt.ptrace(psi, [i, j])
    return float(qt.concurrence(rho_ij))


def witness_W(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    cache: Optional[Dict[Tuple[str, int, int], qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    W_ij(t) = |<sx_i sx_j>| + |<sy_i sy_j>| + |<sz_i sz_j>|
    (bare correlators; NOT connected)
    """
    sx, sy, sz = qt.sigmax(), qt.sigmay(), qt.sigmaz()
    out: Dict[Tuple[int, int], float] = {}

    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = np.nan
            continue

        def get(tag: str, A: qt.Qobj, B: qt.Qobj) -> qt.Qobj:
            if cache is None:
                return two_site_op(A, B, N, i, j)
            return cache[(tag, i, j)]

        ex = qt.expect(get("xx", sx, sx), psi).real
        ey = qt.expect(get("yy", sy, sy), psi).real
        ez = qt.expect(get("zz", sz, sz), psi).real

        out[(i, j)] = float(abs(ex) + abs(ey) + abs(ez))

    return out


# -----------------------------
# Simulation + CSV output
# -----------------------------

def run_and_save_csv(
    N: int,
    Jz: float,
    Jx: float,
    alpha: float,
    times: np.ndarray,
    prefix: str="data",
    outdir: str = "data_qutip",
    psi0: Optional[qt.Qobj] = None,
    bitstring: Optional[str] = None,
    pmstring: Optional[str] = None,
    superpose_spec: Optional[str] = None,
    corr_pairs: Optional[List[Tuple[int, int]]] = None,
    conc_pairs: Optional[List[Tuple[int, int]]] = None,
    wit_pairs: Optional[List[Tuple[int, int]]] = None,
    corrzz_pairs: Optional[List[Tuple[int, int]]] = None,
    do_corrzz: bool = True,
    do_corr: bool = True,
    do_conc: bool = True,
    do_wit: bool = True,
    do_mx: bool = True,
    do_mz: bool = False,
    kac: bool = True,
) -> qt.Qobj:

    os.makedirs(outdir, exist_ok=True)
    times = np.asarray(times, dtype=float)

    # Build Hamiltonian
    H = build_H(N, Jz, Jx, alpha, kac=kac)

    #H = build_H_eff(N, Jz, Jx, alpha, kac=kac, periodic_nn=False)
    # H = build_Heff(N, Jz, Jx, alpha, kac=kac, periodic_nn=False)
    # Choose initial state
    if psi0 is None:
        if superpose_spec is not None:
            spec = superpose_spec.strip().replace(" ", "")
            if "|" not in spec:
                raise ValueError("superpose_spec must look like '<spec1>|<spec2>'")
            s1, s2 = spec.split("|", 1)

            def build_from_spec(s: str) -> qt.Qobj:
                if set(s) <= set("01"):
                    if len(s) != N:
                        raise ValueError(f"bitstring length {len(s)} must equal N={N}.")
                    return product_state_from_bitstring(s)
                if set(s) <= set("+-"):
                    if len(s) != N:
                        raise ValueError(f"pmstring length {len(s)} must equal N={N}.")
                    return product_state_x_pm(s)
                raise ValueError("Each spec must be only '0/1' or only '+/-'.")

            psi0 = superpose(build_from_spec(s1), build_from_spec(s2), normalize=True)

        elif pmstring is not None:
            s = pmstring.strip().replace(" ", "")
            if len(s) != N:
                raise ValueError(f"pmstring length {len(s)} must equal N={N}.")
            psi0 = product_state_x_pm(s)

        elif bitstring is not None:
            s = bitstring.strip().replace(" ", "")
            if len(s) != N:
                raise ValueError(f"bitstring length {len(s)} must equal N={N}.")
            psi0 = product_state_from_bitstring(s)

        else:
            raise ValueError("Provide psi0, or bitstring, or pmstring, or superpose_spec.")

    # Enforce consistent dims (helps ptrace/concurrence)
    psi0 = qt.Qobj(psi0.full(), dims=[[2] * N, [1] * N]).unit()

    # Evolve
    states = evolve_state(H, psi0, times)

    # Pairs + flags
    corr_pairs = corr_pairs or []
    conc_pairs = conc_pairs or []
    wit_pairs  = wit_pairs  or []
    corrzz_pairs = corrzz_pairs or []

    do_corrzz = bool(do_corrzz and len(corrzz_pairs) > 0)
    do_corr = bool(do_corr and len(corr_pairs) > 0)
    do_conc = bool(do_conc and len(conc_pairs) > 0)
    do_wit  = bool(do_wit  and len(wit_pairs)  > 0)

    # Cache 1-site ops
    sx, sz = qt.sigmax(), qt.sigmaz()
    sx_ops = [op_on_site(sx, N, i) for i in range(N)]
    sz_ops = [op_on_site(sz, N, i) for i in range(N)]

    # Cache xx ops for corr (skip self)
    xx_ops = None
    if do_corr:
        xx_ops = {(i, j): two_site_op(sx, sx, N, i, j) for (i, j) in corr_pairs if i != j}

    # Cache zz ops for corrzz (skip self)
    zz_ops = None
    if do_corrzz:
        zz_ops = {(i, j): two_site_op(sz, sz, N, i, j) for (i, j) in corrzz_pairs if i != j}

    # Cache witness ops (xx,yy,zz) for witness pairs (skip self)
    wit_cache = None
    if do_wit:
        sy = qt.sigmay()
        wit_cache = {}
        for (i, j) in wit_pairs:
            if i == j:
                continue
            wit_cache[("xx", i, j)] = two_site_op(sx, sx, N, i, j)
            wit_cache[("yy", i, j)] = two_site_op(sy, sy, N, i, j)
            wit_cache[("zz", i, j)] = two_site_op(sz, sz, N, i, j)

    # Rows
    mx_rows: List[List[float]] = []
    mz_rows: List[List[float]] = []
    corr_rows: List[List[float]] = []
    conc_rows: List[List[float]] = []
    wit_rows:  List[List[float]] = []
    corrzz_rows: List[List[float]] = []

    for t, psi in zip(times, states):
        if do_mx:
            mx = magnetization_x(psi, N, sx_ops=sx_ops)
            mx_rows.append([float(t)] + [float(x) for x in mx])

        if do_mz:
            mz = magnetization_z(psi, N, sz_ops=sz_ops)
            mz_rows.append([float(t)] + [float(z) for z in mz])

        if do_corr:
            corr = correlations_xx_connected(psi, N, corr_pairs, xx_ops=xx_ops, sx_ops=sx_ops)
            corr_rows.append([float(t)] + [float(corr[p]) for p in corr_pairs])

        if do_conc:
            conc = [concurrence_pair(psi, i, j) for (i, j) in conc_pairs]
            conc_rows.append([float(t)] + [float(x) for x in conc])

        if do_wit:
            W = witness_W(psi, N, wit_pairs, cache=wit_cache)
            wit_rows.append([float(t)] + [float(W[p]) for p in wit_pairs])
        
        if do_corrzz:
            corrzz = correlations_zz_connected(psi, N, corrzz_pairs, zz_ops=zz_ops, sz_ops=sz_ops)
            corrzz_rows.append([float(t)] + [float(corrzz[p]) for p in corrzz_pairs])

    # Save
    if do_mx:
        pd.DataFrame(mx_rows, columns=["t"] + [f"mx_{i}" for i in range(N)]).to_csv(
            os.path.join(outdir, f"{prefix}magx.csv"), index=False
        )

    if do_mz:
        pd.DataFrame(mz_rows, columns=["t"] + [f"mz_{i}" for i in range(N)]).to_csv(
            os.path.join(outdir, f"{prefix}magz.csv"), index=False
        )

    if do_corr:
        pd.DataFrame(corr_rows, columns=["t"] + [f"Cxx_{i}_{j}" for (i, j) in corr_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}corr.csv"), index=False
        )

    if do_conc:
        pd.DataFrame(conc_rows, columns=["t"] + [f"C_{i}_{j}" for (i, j) in conc_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}conc.csv"), index=False
        )

    if do_wit:
        pd.DataFrame(wit_rows, columns=["t"] + [f"W_{i}_{j}" for (i, j) in wit_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}wit.csv"), index=False
        )
    
    if do_corrzz:
        pd.DataFrame(corrzz_rows, columns=["t"] + [f"Czz_{i}_{j}" for (i, j) in corrzz_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}corrzz.csv"), index=False
        )

    return H


In [5]:
N = 14
Jz = 1.0
Jx=5
# print(f"{Jx*10:.0f}")
alpha = 1.9
i0=1
tmax=2
tInterval=100
times = np.linspace(0.0, tmax, tInterval+1)
outdir="ED_Data/Trial"
prefix=f"confinement"

# Example: only magnetization (comment out corr/conc cleanly via flags)
#bitstring="00101"
# pmstring="+-+++"
# superpose_spec="00100|00010"
# superpose_spec="+-+-+|00101"
# Bitstring must have length N
H = run_and_save_csv(
    N=N, Jz=Jz, Jx=Jx, alpha=alpha,
    times=times,
    prefix=prefix,
    outdir=outdir,
    # bitstring="10111111111111",
    pmstring="+-++++++++++++",
    # superpose_spec="-+------------|--+-----------",
    # superpose_spec="10111111111111|11011111111111",
    #pmstring="-+--------",
    corr_pairs=[(i0, j) for j in range(N)],     # includes self
    corrzz_pairs=[(i0, j) for j in range(N)], 
    conc_pairs=[(i0, j) for j in range(N)],
    #conc_pairs=[(i, j) for i in range(N) for j in range(N)],
wit_pairs=[(i0, j) for j in range(N)],  # witness skip self
do_corr=True, do_conc=False, do_wit=False, do_corrzz=True,
do_mx=True, do_mz=True,
kac=False,)
print(f"Saved CSVs {Jx*10:.0f} to {outdir}/")

Saved CSVs 50 to ED_Data/Trial/


## Full Version

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

JxValues=np.linspace(0,4,11)
for Jx in JxValues:
    N = 10
    Jz = 1.0
    # Jx
    print(f"{Jx*10:.0f}")
    alpha = 100.5
    i0=1
    tmax=1
    tInterval=100
    times = np.linspace(0.0, tmax, tInterval+1)
    outdir="ED_Data/Trial"
    prefix=f"LRb2alpha{Jx*10:.0f}"

    # Example: only magnetization (comment out corr/conc cleanly via flags)
    #bitstring="00101"
    # pmstring="+-+++"
    # superpose_spec="00100|00010"
    # superpose_spec="+-+-+|00101"
    # Bitstring must have length N
    H = run_and_save_csv(
        N=N, Jz=Jz, Jx=Jx, alpha=alpha,
        times=times,
        prefix=prefix,
        outdir=outdir,
        #bitstring="1011111111",
        # pmstring="+-++++++++",
        superpose_spec="-+------------|--+-----------",
        # superpose_spec="10111111111011|11011111111011",
        #pmstring="-+--------",
        corr_pairs=[(i0, j) for j in range(N)],     # includes self
        conc_pairs=[(i0, j) for j in range(N)],
        #conc_pairs=[(i, j) for i in range(N) for j in range(N)],
    wit_pairs=[(i0, j) for j in range(N)],  # witness skip self
    do_corr=True, do_conc=True, do_wit=True,
    do_mx=True, do_mz=True,
    kac=True,)
    print(f"Saved CSVs {Jx*10:.0f} to {outdir}/")

0
Saved CSVs 0 to ED_Data/ProjectedJxSweep/
4
Saved CSVs 4 to ED_Data/ProjectedJxSweep/
8
Saved CSVs 8 to ED_Data/ProjectedJxSweep/
12
Saved CSVs 12 to ED_Data/ProjectedJxSweep/
16
Saved CSVs 16 to ED_Data/ProjectedJxSweep/
20
Saved CSVs 20 to ED_Data/ProjectedJxSweep/
24
Saved CSVs 24 to ED_Data/ProjectedJxSweep/
28
Saved CSVs 28 to ED_Data/ProjectedJxSweep/
32
Saved CSVs 32 to ED_Data/ProjectedJxSweep/
36
Saved CSVs 36 to ED_Data/ProjectedJxSweep/
40
Saved CSVs 40 to ED_Data/ProjectedJxSweep/


## Projected version

In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

JzValues = np.linspace(0, 5, 41)  # -> [0.0, 2.0]
for Jz in JzValues:
    N = 14
    Jx = 1.0
    print(f"{Jz*10:.0f}")

    alpha = 0.5
    i0 = 1
    tmax = 1
    tInterval = 100
    times = np.linspace(0.0, tmax, tInterval + 1)

    outdir = "ED_Data/ProjectedJzSweep"
    prefix = f"LRb1{Jz*10:.0f}"

    H = run_and_save_csv(
        N=N, Jz=Jz, Jx=Jx, alpha=alpha,
        times=times,
        prefix=prefix,
        outdir=outdir,

        # Make sure each bitstring here has length N=14
        superpose_spec="10111111111101|11011111111101",

        # connected xx
        corr_pairs=[(i0, j) for j in range(N)],

        # connected zz (NEW)
        corrzz_pairs=[(i0, j) for j in range(N)],

        # concurrence pairs
        conc_pairs=[(i0, j) for j in range(N)],

        # witness pairs
        wit_pairs=[(i0, j) for j in range(N)],

        do_corr=True,
        do_corrzz=True,
        do_conc=True,
        do_wit=True,
        do_mx=True,
        do_mz=True,

        kac=False,
    )

    print(f"Saved CSVs {Jx*10:.0f} to {outdir}/")


0
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
1
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
2
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
4
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
5
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
6
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
8
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
9
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
10
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
11
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
12
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
14
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
15
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
16
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
18
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
19
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
20
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
21
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
22
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
24
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
25
Saved CSVs 10 to ED_Data/ProjectedJzSweep/
26
Saved CSVs 10 to ED_Data/ProjectedJzSwe

In [ ]:
N = 14
Jz = 1
Jx=4
print(f"{Jx*10:.0f}")

alpha = 0.5
i0 = 1
tmax = 2
tInterval = 100
times = np.linspace(0.0, tmax, tInterval + 1)

outdir = "ED_Data/zAlignement"
prefix = f"b1z"

H = run_and_save_csv(
    N=N, Jz=Jz, Jx=Jx, alpha=alpha,
    times=times,
    prefix=prefix,
    outdir=outdir,

    # Make sure each bitstring here has length N=14
    #superpose_spec="10111111111011|11011111111011",
    bitstring="10111111111111",
    # connected xx
    corr_pairs=[(i0, j) for j in range(N)],

    # connected zz (NEW)
    corrzz_pairs=[(i0, j) for j in range(N)],

    # concurrence pairs
    conc_pairs=[(i0, j) for j in range(N)],

    # witness pairs
    wit_pairs=[(i0, j) for j in range(N)],

    do_corr=True,
    do_corrzz=True,
    do_conc=True,
    do_wit=True,
    do_mx=True,
    do_mz=True,

    kac=False,
)

print(f"Saved CSVs {Jx*10:.0f} to {outdir}/")

40
Saved CSVs 40 to ED_Data/Trial/


## Random initial state

In [27]:
N = 14
Jz = 1
Jx = 4
print(f"{Jx*10:.0f}")

alpha = 0.5
i0 = 1
tmax = 1
tInterval = 100
times = np.linspace(0.0, tmax, tInterval + 1)

outdir = "ED_Data/Projected"
prefix = f"initvary1"

# --- NEW: init state = cos(theta)|2> + sin(theta) e^{i phi} |3> ---
theta = np.pi/4# #np.pi/2# / 4                    # choose; pi/4 = equal weights
phi   = 0#np.pi #2*np.pi*     # random phase each run

# psi0 = psi_23_superposition(N, theta=theta, phi=phi, site2=1, site3=2)  # sites 1,2 are "2,3" in 1-indexing

psi0 = two_excitation_superposition(
    N=N,
    site_fixed=12,   # <-- fixed flipped site
    site2=1,         # |2>
    site3=2,         # |3>
    theta=theta,
    phi=phi,
    pm=-1,           # try -1 as well
)


H = run_and_save_csv(
    N=N, Jz=Jz, Jx=Jx, alpha=alpha,
    times=times,
    prefix=prefix,
    outdir=outdir,

    psi0=psi0,   # <-- use this
    # superpose_spec="10111111111011|11011111111011",  # <-- remove / comment out

    # connected xx
    corr_pairs=[(i0, j) for j in range(N)],

    # connected zz
    corrzz_pairs=[(i0, j) for j in range(N)],

    # concurrence pairs
    conc_pairs=[(i0, j) for j in range(N)],

    # witness pairs
    wit_pairs=[(i0, j) for j in range(N)],

    do_corr=True,
    do_corrzz=True,
    do_conc=True,
    do_wit=True,
    do_mx=True,
    do_mz=True,
    kac=False,
)

print(f"Saved CSVs {Jx*10:.0f} to {outdir}/")


40
Saved CSVs 40 to ED_Data/Projected/


In [21]:
np.sin(np.pi/4)

np.float64(0.7071067811865476)